# SME Voice Assistant — Live Demo Inference
**Karan Homayounfar | UWE Bristol MSc Data Science | IGP**

Shows vanilla Phi-3 failing vs fine-tuned Phi-3 producing correct JSON — same three inputs, same model, just with and without the QLoRA adapter.

**Before running:**
- GPU: T4 x1 (Notebook settings → Accelerator)
- Datasets added: `sme-phi3-adapter` (adapter weights from training)
- No HF token needed for Phi-3

In [ ]:
import os, json, time, textwrap, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable GPU in settings')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
!pip install -q "transformers==4.45.2" "peft>=0.11.0" "bitsandbytes==0.45.5" accelerate

In [ ]:
# Three demo inputs — booking, cancellation, ambiguous date
DEMO_INPUTS = [
    {
        "label": "Booking request",
        "input": "Hi, I'd like to book a general appointment for next Monday at 10am please.",
        "entities": '{"DATE": "next Monday", "TIME": "10am", "SERVICE": "general appointment"}',
        "expected_action": "book_appointment"
    },
    {
        "label": "Cancellation request",
        "input": "I need to cancel my appointment on Thursday.",
        "entities": '{"DATE": "Thursday"}',
        "expected_action": "cancel_appointment"
    },
    {
        "label": "Availability check",
        "input": "Do you have anything free this Friday afternoon for a dental check-up?",
        "entities": '{"DATE": "this Friday", "TIME": "afternoon", "SERVICE": "dental check-up"}',
        "expected_action": "check_availability"
    },
]

SYSTEM_PROMPT = (
    "You are an appointment booking assistant. "
    "Given a caller utterance and extracted entities, respond with a single valid JSON object. "
    "The JSON must have an 'action' field (one of: book_appointment, cancel_appointment, "
    "check_availability, confirm, end_call, unknown) and relevant fields for that action. "
    "Output only the JSON object, nothing else."
)

def make_prompt(demo):
    user_msg = f"Utterance: {demo['input']}\nEntities: {demo['entities']}"
    return (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{user_msg}<|end|>\n"
        f"<|assistant|>\n"
    )

print('Demo inputs ready.')

In [ ]:
PHI3_ID      = 'microsoft/Phi-3-mini-4k-instruct'
ADAPTER_PATH = '/kaggle/input/sme-phi3-adapter'

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('Loading Phi-3 mini base model...')
tokenizer = AutoTokenizer.from_pretrained(PHI3_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    PHI3_ID,
    quantization_config=bnb_cfg,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation='eager',
)
base_model.eval()
print('Base model loaded.')

In [ ]:
def run_inference(model, prompt, max_new=80):
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    input_len = inputs['input_ids'].shape[1]
    eos_id = tokenizer.convert_tokens_to_ids('<|end|>')
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=eos_id,
        )
    latency_ms = (time.perf_counter() - t0) * 1000
    text = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
    return text, round(latency_ms)

def parse_action(text):
    s, e = text.find('{'), text.rfind('}')
    if s == -1 or e == -1:
        return None
    try:
        return json.loads(text[s:e+1])
    except Exception:
        return None

def show_result(label, raw, latency_ms, expected_action):
    parsed = parse_action(raw)
    if parsed:
        action = parsed.get('action', '???')
        correct = action == expected_action
        status = 'CORRECT' if correct else 'WRONG ACTION'
        print(f'  Output : {json.dumps(parsed, indent=2)}')
        print(f'  Action : {action}  [{status}]')
    else:
        print(f'  Output : {repr(raw[:120])}')
        print(f'  Action : [INVALID JSON - no parseable output]')
    print(f'  Latency: {latency_ms}ms')

print('Inference helpers ready.')

## Vanilla Phi-3 (no fine-tuning)

In [ ]:
print('=' * 60)
print('VANILLA PHI-3 MINI  (no adapter, no fine-tuning)')
print('=' * 60)

vanilla_results = []
for demo in DEMO_INPUTS:
    print(f"\n[{demo['label']}]")
    print(f"  Input  : {demo['input']}")
    prompt = make_prompt(demo)
    raw, latency = run_inference(base_model, prompt)
    show_result(demo['label'], raw, latency, demo['expected_action'])
    vanilla_results.append(raw)

print('\nVanilla inference done.')

## Fine-tuned Phi-3 (QLoRA adapter, 600 synthetic samples, ~70 min on T4)

In [ ]:
print('Loading QLoRA adapter...')
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
ft_model.eval()
print('Adapter loaded.')

In [ ]:
print('=' * 60)
print('FINE-TUNED PHI-3 MINI  (QLoRA r=16, 600 samples, Kaggle T4)')
print('=' * 60)

for demo in DEMO_INPUTS:
    print(f"\n[{demo['label']}]")
    print(f"  Input  : {demo['input']}")
    prompt = make_prompt(demo)
    raw, latency = run_inference(ft_model, prompt)
    show_result(demo['label'], raw, latency, demo['expected_action'])

print('\nFine-tuned inference done.')

## Summary

In [ ]:
print('=' * 60)
print('EVALUATION RESULTS  (480-sample hold-out set)')
print('=' * 60)
print(f"{'Model':<30} {'Action Acc':>12} {'Exact Match':>12}")
print('-' * 56)
rows = [
    ('Phi-3 mini (vanilla)',      '0.4%',   '0.0%'),
    ('Phi-3 mini (fine-tuned)',   '98.1%',  '70.6%'),
    ('Llama 3.2 3B (vanilla)',    '0.0%',   '0.0%'),
    ('Llama 3.2 3B (fine-tuned)', '99.8%',  '70.4%'),
]
for model, acc, em in rows:
    print(f"{model:<30} {acc:>12} {em:>12}")
print()
print('H1 (fine-tuned > 90%):   CONFIRMED')
print('H2 (vanilla < 5%):       CONFIRMED')
print('H3 (Llama > Phi-3 FT):   CONFIRMED')